# Stage 2 Notebook 50 - Exp2UU K=64 query head + VFL + full 70K dataset

**Validate NB47 at scale.** NB47 with K=64 queries + Hungarian + VFL was the first cls-discriminating experiment (pos-neg gap=0.099, val_lane_f1=0.246, val_lane_best_f1=0.344) but its geometry was capped at matched_iou=0.27 because 3000 samples isn't enough to ground 64 free-form queries. NB48 showed full 70K data lifts the anchor head's matched_iou from 0.525 to 0.544 -- a similar lift is plausible for K=64 queries.

Single config diff vs NB47:
- `train.end_epoch: 20 -> 6` (~7x more total iters from full data)
- LIMIT_TRAIN flag REMOVED -> full 70K split
- All else identical to NB47 (exp42 yaml).

Wall-clock ~ 60-80 minutes on RTX Pro 6000.

### Run mode

1. `DEBUG_MODE = True` smoke.
2. `DEBUG_MODE = False` for full-dataset 6-epoch run.
3. Wall-clock ~ 60-80 min.
4. Independent of all prior NBs.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp45_rmt_gca_query64_vfl_full_dataset_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp45_rmt_gca_query64_vfl_full_dataset_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp45_rmt_gca_query64_vfl_full_dataset_joint_smoke.log
OK exp45_rmt_gca_query64_vfl_full_dataset_joint.yaml
  lane_shape=(1, 64, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.1803 det_loss=3.2145 grad_cos=0.1230 lambda_lane=0.0565
  gate_stats={'gate/det_mean': 0.4973973333835602, 'gate/lane_mean': 0.5026984810829163, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp45_rmt_gca_query64_vfl_full_dataset_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full6'
    EPOCHS = 6
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp45_rmt_gca_query64_vfl_full_dataset_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp45_rmt_gca_query64_vfl_full_dataset_joint_full6 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp45_rmt_gca_query64_vfl_full_dataset_joint_full6.tar --epochs 6 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp45_rmt_gca_query64_vfl_full_dataset_joint_full6.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp45_rmt_gca_query64_vfl_full_dataset_joint_full6_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp45_rmt_gca_query64_vfl_full_dataset_joint.yaml --curve-tar /conten

0

## What to watch in Exp2UU training

Reference NB47 (K=64 + 3K data): val_lane_f1=0.246, val_lane_best_f1=0.344, matched_iou=0.27.

Pass criteria at epoch 6 on full 70K:
- **`val/matched_line_iou >= 0.40`** -- 1.5x NB47's 0.27, similar lift to what NB48 showed for the anchor head.
- **`val/lane_f1 >= 0.30`** -- 1.2x NB47's 0.246, with 23x more data variety.
- **`val/lane_best_f1 >= 0.40`**.
- **`val/lane/decoded_f1 >= 0.10`** -- 5x NB47's 0.019, because better geometry now multiplies with the already-working cls.
- `val/lane/decoded_oracle_f1 >= 0.30`.
- pos-neg gap should stay >= 0.05 (don't lose the cls breakthrough).

Failure signals:
- matched_iou < 0.30: geometry didn't scale with data. Switch to Exp2TT's hybrid architecture.
- val_det stalls like NB48 (>= 3.0): joint conflict at full data scale; need Exp2VV's lambda_det=2.0 fix.